In [39]:
import sys
import os
import urllib3
from configparser import ConfigParser

# Add your local ThreatConnect SDK to path
sys.path.append("Z:/HTOC/Data_Analytics/threatconnect")
from ThreatConnect import ThreatConnect
from RequestObject import RequestObject
from Owners import Owners

# Add your project repo to path
project_root = r"C:\Users\jaskew\Documents\project_repository\scripts\Data Movement\ThrearConnect-api-pull"
if project_root not in sys.path:
    sys.path.append(project_root)

from utils.config_loader import load_config

# Load API config
config_path = os.path.join(project_root, "utils", "config.json")
try:
    api_secret_key, api_access_id, api_base_url, api_default_org = load_config(config_path)
    print(f"Loaded config from: {config_path}")
    print(f"Base URL: {api_base_url}")
    print(f"Access ID: {api_access_id}")
    print(f"Default Org: {api_default_org}")
except Exception as e:
    print(f"[ERROR] Failed to load configuration: {e}")
    sys.exit(1)

# Disable SSL verification warnings (use cautiously)
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
verify_ssl = False

# Initialize ThreatConnect session
try:
    tc = ThreatConnect(api_access_id, api_secret_key, api_default_org, api_base_url)
    print("ThreatConnect initialized.")
except Exception as e:
    print(f"[ERROR] Failed to initialize ThreatConnect: {e}")
    sys.exit(1)

# Define the owner (organization scope)
owner = 'HTOC Org'

# Create a request object to fetch indicators (or other data)
try:
    ro = RequestObject()
    ro.set_http_method('GET')
    ro.set_owner(owner)
    ro.set_owner_allowed(True)
    # ro.set_resource_pagination(True)  # Uncomment if needed
    print("RequestObject successfully created.")
except Exception as e:
    print(f"[ERROR] Failed to initialize RequestObject: {e}")
    sys.exit(1)



Loaded config from: C:\Users\jaskew\Documents\project_repository\scripts\Data Movement\ThrearConnect-api-pull\utils\config.json
Base URL: https://hvs.threatconnect.com/api
Access ID: 09783848890162390382
Default Org: HTOC Org
ThreatConnect initialized.
RequestObject successfully created.


In [40]:
import pandas as pd
from datetime import datetime, timedelta
import urllib.parse
import pytz
from collections import defaultdict

# --- ThreatConnect setup (assumes 'tc' and 'ro' objects already exist) ---
# Make sure tc and ro are defined in your environment based on your SDK usage

# Define the time window (3 days ago at 00:00 UTC)
start_date = (datetime.now(pytz.UTC) - timedelta(days=3)).date()
start = f"{start_date}T00:00:00Z"

# Indicator types to query
type_names = [
    "Address", "EmailAddress", "File", "Host", "URL", "ASN", "CIDR",
    "Email Subject", "Hashtag", "Mutex", "Registry Key", "Stripped URL", "User Agent"
]
type_name_condition = ", ".join([f'"{t}"' for t in type_names])

# Owners to pull from
list_of_owners = ['HTOC Org']
final_results = []

for owner in list_of_owners:
    print(f"\nQuerying owner: {owner}")
    try:
        tql_raw = (
            f'ownerName EQ "{owner}" AND '
            f'typeName IN ({type_name_condition}) AND '
            f'lastObserved >= "{start}"'
        )
        tql_encoded = urllib.parse.quote(tql_raw)

        ro.set_http_method('GET')
        ro.set_request_uri(
            f'/v3/indicators?tql={tql_encoded}&fields=tags,observations&resultStart=0&resultLimit=10000'
        )

        response = tc.api_request(ro)

        if response.headers.get('content-type') == 'application/json':
            results = response.json()
            final_results.append(results)
        else:
            print(f"Unexpected response format: {response.headers.get('content-type')}")
    except Exception as e:
        print(f"Failed to query indicators for {owner}: {e}")

# Normalize and clean the indicators
if final_results:
    normalized_data = []
    for result in final_results:
        if 'data' in result:
            for item in result['data']:
                if 'summary' in item:
                    normalized_data.append(item)

    if normalized_data:
        observed_src = pd.json_normalize(normalized_data)
        observed_src['indicator'] = observed_src['summary'].str.split(' ', expand=True)[0].str.strip()
        observed_src = observed_src.drop_duplicates(subset='indicator', keep='first')

        # Ensure lastObserved is a datetime object for filtering
        observed_src['lastObserved'] = pd.to_datetime(observed_src['lastObserved'], utc=True)
        start_dt = pd.to_datetime(start)
        observed_src = observed_src[observed_src['lastObserved'] >= start_dt]

        print(f"\nRetrieved {len(observed_src)} unique observed indicators.")

        # -------------------------------
        # Enrich indicators with VirusTotalV3
        # -------------------------------
        indicator_ids = observed_src['id'].dropna().unique().tolist()
        enriched_results = []

        print(f"\nEnriching {len(indicator_ids)} indicators with VirusTotalV3...")

        for indicator_id in indicator_ids:
            try:
                enrich_url = f'/v3/indicators/{indicator_id}/enrich?type=VirusTotalV3'
                ro.set_http_method('POST')
                ro.set_request_uri(enrich_url)
                ro.set_body({})  # Enrichment call usually has no body

                enrich_response = tc.api_request(ro)

                if enrich_response.status_code == 200:
                    enrich_data = enrich_response.json()
                    enrich_data['indicator_id'] = indicator_id
                    enriched_results.append(enrich_data)

            except Exception as e:
                continue

        # Convert enriched data into DataFrame
        if enriched_results:
            df_enriched = pd.json_normalize(enriched_results)
            df_enriched.rename(columns={'indicator_id': 'id'}, inplace=True)

            # Merge enrichment data into main dataframe
            observed_src = observed_src.merge(df_enriched, on='id', how='left')
            print(f"\nSuccessfully enriched and merged {len(df_enriched)} indicators.")
        else:
            print("\nNo enrichment data retrieved.")

    else:
        print("\nNo valid indicators with 'summary' key retrieved.")
else:
    print("\nNo indicators retrieved.")




Querying owner: HTOC Org

Retrieved 860 unique observed indicators.

Enriching 860 indicators with VirusTotalV3...


Status Code: 400
Failed API Response: b'{"errCode":"0x1001","message":"The EmailAddress kathy@finnaccounting.org cannot be enriched with VirusTotal because the indicator type isn\'t supported.","status":"Error"}'
Status Code: 400
Failed API Response: b'{"errCode":"0x1001","message":"The EmailAddress ginny.williams@frcohio.org cannot be enriched with VirusTotal because the indicator type isn\'t supported.","status":"Error"}'
Status Code: 400
Failed API Response: b'{"errCode":"0x1001","message":"The EmailAddress cameron@yourpensionmeeting.com cannot be enriched with VirusTotal because the indicator type isn\'t supported.","status":"Error"}'
Status Code: 400
Failed API Response: b'{"errCode":"0x1001","message":"The EmailAddress no-reply@thryv.com cannot be enriched with VirusTotal because the indicator type isn\'t supported.","status":"Error"}'
Status Code: 400
Failed API Response: b'{"errCode":"0x1001","message":"The EmailAddress k.baker@positiveproximity.com cannot be enriched with Viru


Successfully enriched and merged 836 indicators.


In [41]:
observed_src

,id,dateAdded,ownerId,ownerName,webLink,type,lastModified,rating,confidence,source,...,data.summary,data.privateFlag,data.active,data.activeLocked,data.ip,data.legacyLink,data.enrichment.data,data.hostName,data.dnsActive,data.whoisActive
0,6755399460008447,2025-07-02T12:05:37Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-11-03T14:36:45Z,1.0,56,Treasury,...,38.211.44.18,False,True,False,38.211.44.18,https://hvs.threatconnect.com/auth/indicators/...,"[{'type': 'VirusTotal', 'vtMaliciousCount': 0}]",NaN,NaN,NaN
1,6755399458556973,2025-06-11T14:46:20Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-11-03T14:16:34Z,3.0,60,NaN,...,193.163.125.187,False,True,False,193.163.125.187,https://hvs.threatconnect.com/auth/indicators/...,"[{'type': 'VirusTotal', 'vtMaliciousCount': 5}]",NaN,NaN,NaN
2,5629499555099353,2025-06-23T15:22:17Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-11-03T14:16:34Z,3.0,61,NaN,...,193.163.125.197,False,True,False,193.163.125.197,https://hvs.threatconnect.com/auth/indicators/...,"[{'type': 'VirusTotal', 'vtMaliciousCount': 5}]",NaN,NaN,NaN
3,5629499555099352,2025-06-23T15:22:15Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-11-03T14:16:34Z,3.0,61,NaN,...,193.163.125.136,False,True,False,193.163.125.136,https://hvs.threatconnect.com/auth/indicators/...,"[{'type': 'VirusTotal', 'vtMaliciousCount': 8}]",NaN,NaN,NaN
4,6755399459078675,2025-06-25T17:48:26Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-11-03T14:15:41Z,3.0,62,NaN,...,193.163.125.193,False,True,False,193.163.125.193,https://hvs.threatconnect.com/auth/indicators/...,"[{'type': 'VirusTotal', 'vtMaliciousCount': 7}]",NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
855,5265398,2025-01-23T20:41:54Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Stripped URL,2025-01-31T16:29:51Z,5.0,91,https://careersinpsychology.org,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
856,4303591,2023-03-03T13:53:09Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Stripped URL,2025-01-24T23:25:10Z,3.0,72,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
857,4476331,2023-11-16T13:43:29Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Stripped URL,2025-01-17T23:24:44Z,4.0,85,http://selligenttier.naylorcampaigns.com,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
858,4442727,2023-10-06T23:26:48Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Stripped URL,2024-03-28T23:23:43Z,3.0,83,https://google,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [42]:
# Unnest tags.data in observed_src to get a list of each tag per indicator

# Explode tags.data to one row per tag
tags_exploded = (
    observed_src[['indicator', 'tags.data']]
      .explode('tags.data')
      .dropna(subset=['tags.data'])
)

# Extract tag name from each tag object
tags_exploded['tag_name'] = tags_exploded['tags.data'].apply(lambda x: x.get('name') if isinstance(x, dict) else None)

# Aggregate all tag names into a list per indicator
tags_per_indicator = (
    tags_exploded.groupby('indicator')['tag_name']
    .apply(lambda x: [t for t in x if t])
    .reset_index()
    .rename(columns={'tag_name': 'tag_list'})
)

# Merge back to observed_src if desired
observed_src_with_tags = observed_src.merge(tags_per_indicator, on='indicator', how='left')

display(observed_src_with_tags[['indicator', 'tag_list']])

,indicator,tag_list
0,38.211.44.18,"[INDICATOR NOTICE 25178.1, Mr Hamza Group, T-S..."
1,193.163.125.187,"[DHA Splunk API, OS Splunk API, FDA Splunk API..."
2,193.163.125.197,"[DHA Splunk API, OS Splunk API, FDA Splunk API..."
3,193.163.125.136,"[DHA Splunk API, OS Splunk API, FDA Splunk API..."
4,193.163.125.193,"[DHA Splunk API, OS Splunk API, FDA Splunk API..."
...,...,...
855,careersinpsychology.org,"[Malware Family: GOOTLOADER, Gootloader]"
856,aka.ms/o0ukef,"[FDA Splunk API, Observed, Phishing Email, inv..."
857,selligenttier.naylorcampaigns.com,"[OS Splunk API, VA OIS CSOC CTS Splunk, FDA Sp..."
858,google,"[OS Splunk API, HRSA Splunk API, Observed]"


In [43]:
# List of tags to count (case-insensitive, strip whitespace)
tags_of_interest = [
    "Scanning", "DDoS", "Spam", "Phishing", "Cryptojacking",
    "Credential Stuffing", "Ransomware", "Data Theft",
    "Cross Site Scripting Attacks", "SQL Injections"
]

# Normalize tag_list to lowercase for matching
def tag_counter(tag_list, tag):
    if not isinstance(tag_list, list):
        return 0
    return sum(1 for t in tag_list if isinstance(t, str) and t.strip().lower() == tag.lower())

tag_counts = {}
for tag in tags_of_interest:
    tag_counts[tag] = observed_src_with_tags['tag_list'].apply(lambda lst: tag_counter(lst, tag)).sum()
    # Add a column to observed_src with the list of matching "Botnet" tags per indicator
    botnet_tags = set(tags_of_interest)
    def extract_botnet_tags(tag_list):
        if not isinstance(tag_list, list):
            return []
        return [t for t in tag_list if isinstance(t, str) and t.strip().lower() in {b.lower() for b in botnet_tags}]

    # Map indicator to its tag_list for efficient lookup
    indicator_to_tags = dict(zip(observed_src_with_tags['indicator'], observed_src_with_tags['tag_list']))

    observed_src['Botnet'] = observed_src['indicator'].map(lambda ind: extract_botnet_tags(indicator_to_tags.get(ind, [])))
# Display as a DataFrame for readability
pd.DataFrame(list(tag_counts.items()), columns=['Tag', 'Count'])

,Tag,Count
0,Scanning,0
1,DDoS,64
2,Spam,0
3,Phishing,10
4,Cryptojacking,0
5,Credential Stuffing,0
6,Ransomware,3
7,Data Theft,11
8,Cross Site Scripting Attacks,0
9,SQL Injections,0


In [44]:
import os
import pandas as pd
from datetime import datetime, timedelta

# Base file path with placeholder for date
base_path = r"Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d{date}.csv"
#base_path = r"C:\Users\jaskew\Documents\project_repository\data\raw\ObservationDataFiles\htoc_opdiv_obs_d{date}.csv"
date_format = "%Y%m%d"

def get_file_paths(base_path, days=3):
    """ Generate file paths for the last `days` days using list comprehension. """
    today = datetime.utcnow()
    dates_to_pull = [(today - timedelta(days=i)).strftime(date_format) for i in range(days)]
    
    # Construct file paths
    file_paths = [base_path.format(date=dt) for dt in dates_to_pull]
    
    # Filter for existing files
    existing_files = [file_path for file_path in file_paths if os.path.exists(file_path)]
    
    if not existing_files:
        print("No files found for the specified date range.")
    else:
        print(f"Files to be loaded: {existing_files}")
    
    return existing_files

def load_observed_data(file_paths):
    """ Load and concatenate observed data from multiple files. """
    data_frames = []

    for file_path in file_paths:
        try:
            df = pd.read_csv(file_path)
            data_frames.append(df)
        except Exception as e:
            print(f"Error reading file {file_path}: {e}")
    
    # Concatenate data
    if data_frames:
        observed_data_df = pd.concat(data_frames, ignore_index=True)
        print(f"Loaded data from {len(data_frames)} files.")
    else:
        observed_data_df = pd.DataFrame()

    return observed_data_df

# Example Usage:
# Fetch file paths for the last 3 days
file_paths = get_file_paths(base_path, days=5)

# Load observed data
observed_data_df = load_observed_data(file_paths)



C:\Users\jaskew\AppData\Local\Temp\ipykernel_640\1651825034.py:12: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  today = datetime.utcnow()


Files to be loaded: ['Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20251103.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20251102.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20251101.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20251031.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20251030.csv']
Loaded data from 5 files.


In [45]:
import pandas as pd

df = observed_src.copy()

# explode tags.data
tags_exploded = (
    df[['indicator', 'tags.data']]
      .explode('tags.data')
      .dropna(subset=['tags.data'])
)

# normalize all fields in tags.data into flat columns
tags_norm = pd.json_normalize(tags_exploded['tags.data'])
tags_norm.columns = [f"tag_{c}" for c in tags_norm.columns]

# re‑attach indicator
tags_expanded = tags_exploded.reset_index(drop=True).join(tags_norm)

# extract partners
tags_expanded['partner'] = tags_expanded['tag_name'].map(
    lambda n: n[:-len(' Splunk API')] if isinstance(n, str) and n.endswith(' Splunk API') else None
)

# aggregate each tag_* field into a list per indicator
tag_fields = list(tags_norm.columns)
tag_agg = (
    tags_expanded
      .groupby('indicator', as_index=False)
      .agg({
          **{f: list for f in tag_fields},
          'partner': lambda x: [p for p in dict.fromkeys(x) if p]
      })
      .rename(columns={'partner':'partners'})
)


# --- CORE AGGREGATION OF OTHER COLUMNS ---

first_cols = [
    'id','dateAdded','ownerId','ownerName','webLink','type','lastModified', 'falsePositives',
    'rating','confidence','description','summary','observations',
    'lastObserved','privateFlag','active','activeLocked','ip',
    'legacyLink','source','address','url'
]
# Remove 'hostName', 'dnsActive', 'whoisActive' from first_cols to avoid KeyError

# Add 'Botnet' column from observed_src if it exists
if 'Botnet' in observed_src.columns:
    df['Botnet'] = observed_src['Botnet']
    first_cols.append('Botnet')

base_agg = (
    df
      .drop(columns=[
          'createdBy.id','createdBy.userName','createdBy.firstName','createdBy.lastName',
          'createdBy.pseudonym','createdBy.owner','xid','eventType','documentDateAdded',
          'documentType','fileSize','fileName','downVoteCount','upVoteCount','type_group',
          'webLink_group','ownerName_group','ownerId_group','dateAdded_group','id_group',
          'platforms.count','tactics.count',
      ], errors='ignore')
      .groupby('indicator', as_index=False)[
          [c for c in first_cols if c in df.columns]
      ]
      .first()
)

# --- MERGE EVERYTHING BACK ---

agg_df = (
    base_agg
      .merge(tag_agg,   on='indicator', how='left')
#      .merge(group_agg, on='indicator', how='left')
)

def clean_list(lst):
    if not isinstance(lst, list):
        return []
    cleaned = []
    for v in lst:
        # drop any null‑like values
        try:
            if pd.isna(v):
                continue
        except Exception:
            pass
        # drop empty strings
        if isinstance(v, str) and v == "":
            continue
        cleaned.append(v)
    return cleaned

def list_to_csv(lst):
    """
    Takes a cleaned list and returns:
      - a comma-separated string of its items, OR
      - an empty string if there are none.
    """
    if not lst:
        return ""
    return ", ".join(str(v) for v in lst)


for col in ['partners', 'group_ids', 'group_names'] + tag_fields:
    if col in agg_df.columns:
        agg_df[col] = agg_df[col].apply(clean_list).apply(list_to_csv)
    
display(agg_df)



,indicator,id,dateAdded,ownerId,ownerName,webLink,type,lastModified,rating,confidence,...,tag_id,tag_name,tag_lastUsed,tag_description,tag_techniqueId,tag_tactics.data,tag_tactics.count,tag_platforms.data,tag_platforms.count,partners
0,101.71.130.99,5629499572136493,2025-10-16T15:41:04Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-11-02T11:13:46Z,3.0,78,...,"471298, 35760, 35057, 30479, 23630, 23576, 749","DHA Splunk API, OS Splunk API, FDA Splunk API,...","2025-11-03T10:08:20Z, 2025-11-03T14:02:40Z, 20...",Observations reported by the HHS Ofice of the ...,,,,,,"DHA, OS, FDA, CMS, NIH"
1,101.89.174.236,5629499542017370,2025-05-21T18:39:43Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-11-03T08:56:35Z,3.0,58,...,"471298, 35760, 35057, 30479, 23667, 23630, 235...","DHA Splunk API, OS Splunk API, FDA Splunk API,...","2025-11-03T10:08:20Z, 2025-11-03T14:02:40Z, 20...",Observations reported by the HHS Ofice of the ...,,,,,,"DHA, OS, FDA, CMS, HRSA, NIH, IHS, HHS"
2,103.101.216.50,6755399460008262,2025-07-02T12:05:36Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-11-03T13:42:03Z,1.0,56,...,"1469320, 1455870, 505200, 471298, 35057, 23576...","INDICATOR NOTICE 25178.1, Mr Hamza Group, T-Su...","2025-07-18T13:37:52Z, 2025-10-21T17:48:35Z, 20...",,,,,,,"DHA, FDA"
3,103.133.60.12,6755399460008266,2025-07-02T12:05:36Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-10-31T10:13:23Z,1.0,57,...,"1469320, 1455870, 505200, 471298, 35057, 23769...","INDICATOR NOTICE 25178.1, Mr Hamza Group, T-Su...","2025-07-18T13:37:52Z, 2025-10-21T17:48:35Z, 20...",,,,,,,"DHA, FDA, HRSA, HHS"
4,103.133.61.182,6755399460007421,2025-07-02T12:05:29Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-11-03T12:59:08Z,1.0,56,...,"1469320, 1455870, 505200, 471298, 35057, 23769...","INDICATOR NOTICE 25178.1, Mr Hamza Group, T-Su...","2025-07-18T13:37:52Z, 2025-10-21T17:48:35Z, 20...",,,,,,,"DHA, FDA, HRSA, NIH"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
855,warlockstallioniso.com,5629499563360176,2025-08-13T15:23:59Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-10-31T02:54:38Z,3.0,59,...,"471298, 38829, 30770, 23819, 23769, 23630, 235...","DHA Splunk API, Phishing, I&W, malicious javas...","2025-11-03T10:08:20Z, 2025-10-29T17:33:06Z, 20...",Adversaries may send phishing messages to gain...,T1566,['Initial Access'],1.0,"['Linux', 'macOS', 'Windows', 'SaaS', 'Identit...",6.0,"DHA, NIH"
856,www.deepseek.com,5271608,2025-01-29T16:27:10Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-10-28T11:02:49Z,3.0,71,...,"471298, 461545, 35760, 35752, 35057, 30770, 30...","DHA Splunk API, DeepSeek, OS Splunk API, VA OI...","2025-11-03T10:08:20Z, 2025-01-31T14:04:11Z, 20...",Observations reported by the HHS Ofice of the ...,,,,,,"DHA, OS, FDA, CMS, HRSA, NIH, IHS"
857,www.deepseek.com.cdn.cloudflare.net,5271612,2025-01-29T16:27:10Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-10-29T11:16:44Z,3.0,69,...,"471298, 461545, 35752, 30479, 23769, 23630, 23...","DHA Splunk API, DeepSeek, VA OIS CSOC CTS Splu...","2025-11-03T10:08:20Z, 2025-01-31T14:04:11Z, 20...",,,,,,,"DHA, CMS, NIH, IHS"
858,www.sthda.com,4565837,2024-04-16T17:33:15Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-10-30T02:36:46Z,4.0,37,...,"471298, 35760, 35752, 35057, 30770, 30479, 237...","DHA Splunk API, OS Splunk API, VA OIS CSOC CTS...","2025-11-03T10:08:20Z, 2025-11-03T14:02:40Z, 20...",Observations reported by the HHS Ofice of the ...,,,,,,"DHA, OS, FDA, CMS, HRSA, NIH, IHS"


In [46]:
import pandas as pd
from datetime import timedelta
from pandas import Timestamp

# Setup cutoff timestamps
cutoff = Timestamp.utcnow()
date_added_cutoff = cutoff - timedelta(days=30)

# Initialize filtered tags DataFrame
filtered_tags = pd.DataFrame()

# Filter API tags and attach metadata
for _, row in observed_src.iterrows():
    tags_data = row.get('tags.data')

    if isinstance(tags_data, list):
        tags = pd.json_normalize(tags_data)
        tags['name'] = tags['name'].astype(str)
        all_tags_list = tags['name'].tolist()

        api_tags = tags[tags['name'].str.contains('API', case=False, na=False)].copy()
        if not api_tags.empty:
            metadata_cols = [
                'summary', 'observations', 'description', 'type',
                'dateAdded', 'lastModified', 'lastObserved', 'webLink', 'data.enrichment.data'
            ]
            for col in metadata_cols:
                value = row.get(col)
                # If value is a list and not a string, assign [value] * len(api_tags)
                if isinstance(value, list) and not isinstance(value, str):
                    api_tags[col] = [value] * len(api_tags)
                else:
                    api_tags[col] = value
            # Fix: assign all_tags as a list of lists, one per row
            api_tags['all_tags'] = [all_tags_list] * len(api_tags)
            filtered_tags = pd.concat([filtered_tags, api_tags], ignore_index=True)

# Convert date columns
filtered_tags['lastObserved'] = pd.to_datetime(filtered_tags['lastObserved'], errors='coerce')
filtered_tags['dateAdded'] = pd.to_datetime(filtered_tags['dateAdded'], errors='coerce')

# Validate required columns
required_columns = ['indicator', 'OpDiv', 'obs_date']
missing_columns = [col for col in required_columns if col not in observed_data_df.columns]
if missing_columns:
    raise ValueError(f"Missing columns in ProcessedObservedData.csv: {missing_columns}")

# Clean 'name' column and match observations
filtered_tags['cleaned_name'] = filtered_tags['name'].str.replace(r'\s+Splunk API$', '', regex=True)
filtered_tags['observed_date'] = pd.NaT

for index, row in filtered_tags.iterrows():
    match = observed_data_df[
        (observed_data_df['indicator'] == row['summary']) &
        (observed_data_df['OpDiv'] == row['cleaned_name'])
    ]
    if not match.empty:
        filtered_tags.at[index, 'observed_date'] = match['obs_date'].iloc[0]

filtered_tags['observed_date'] = pd.to_datetime(filtered_tags['observed_date'], errors='coerce')
filtered_tags.drop(columns=['cleaned_name'], inplace=True)

# Filter recent tags (last 24 hours)
cutoff_naive = cutoff.tz_localize(None) if cutoff.tzinfo else cutoff
recent_tags = filtered_tags[
    (filtered_tags['lastObserved'] >= cutoff - timedelta(hours=24)) &
    (filtered_tags['observed_date'] >= cutoff_naive - timedelta(days=1))
].copy()

# Extract partner from name and group
recent_tags['partner'] = recent_tags['name'].str.replace(' Splunk API', '', regex=False)

partner_groups = (
    recent_tags.groupby('summary')['partner']
    .agg(['nunique', lambda x: ', '.join(sorted(set(x)))])
    .reset_index()
    .rename(columns={'nunique': 'partner_count', '<lambda_0>': 'partners'})
)

recent_tags = recent_tags.merge(partner_groups, on='summary', how='left')

# Remove duplicate summaries
recent_tags = recent_tags.drop_duplicates(subset='summary', keep='first')

# Unnest 'data.enrichment.data' (list of dicts) into separate columns
if 'data.enrichment.data' in recent_tags.columns:
    enrichment_df = pd.json_normalize(
        recent_tags['data.enrichment.data'].dropna().explode()
    )
    enrichment_df.index = recent_tags['data.enrichment.data'].dropna().explode().index
    enrichment_cols = [col for col in enrichment_df.columns if col not in recent_tags.columns]
    # Join enrichment columns back to recent_tags
    recent_tags = recent_tags.join(enrichment_df[enrichment_cols], how='left')

    # Keep only records with vtMaliciousCount > 10
    recent_tags = recent_tags[recent_tags['vtMaliciousCount'] > 10]
    
# Drop unused columns if present
columns_to_drop = [
    'techniqueId', 'tactics.data', 'tactics.count',
    'platforms.data', 'platforms.count', 'partner', 'name', 'data.enrichment.data'
]
recent_tags.drop(columns=[col for col in columns_to_drop if col in recent_tags.columns], inplace=True, errors='ignore')

# Optional: remove certain tags
# recent_tags = recent_tags[~recent_tags['all_tags'].apply(lambda x: 'I&W' in x or 'htoc_wl' in x if isinstance(x, list) else False)]

recent_tags

,id,lastUsed,description,summary,observations,type,dateAdded,lastModified,lastObserved,webLink,all_tags,observed_date,partner_count,partners,vtMaliciousCount
122,30479,2025-11-02T14:39:01Z,185.220.101[.]182 is registered in Germany und...,185.220.101.182,228072,Address,2021-12-15 13:22:24+00:00,2025-11-03T12:55:25Z,2025-11-03 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[Zero Day, HHS Splunk Production API, HRSA Spl...",2025-11-03,1,CMS,13.0
135,471298,2025-11-03T10:08:20Z,RITM0582833,193.163.125.121,3609434,Address,2025-05-21 13:34:32+00:00,2025-11-03T12:55:04Z,2025-11-03 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, VA OIS CSOC CT...",2025-11-03,7,"CMS, DHA, FDA, HHS, HRSA, IHS, OS",13.0
193,471298,2025-11-03T10:08:20Z,Executive Summary: \n\n118.193.59[.]10 resolv...,118.193.59.10,11325038,Address,2025-02-12 19:49:28+00:00,2025-11-03T12:36:17Z,2025-11-03 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, Active Scanning: Scanning IP ...",2025-11-03,6,"CMS, DHA, FDA, HRSA, IHS, OS",12.0
199,35760,2025-11-03T14:02:40Z,INC9235915,165.154.173.226,577477,Address,2025-09-12 14:57:07+00:00,2025-11-03T12:36:17Z,2025-11-03 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-11-03,4,"CMS, FDA, IHS, OS",11.0
220,471298,2025-11-03T10:08:20Z,TASK0902923,198.235.24.194,15656064,Address,2025-07-28 17:34:11+00:00,2025-11-03T12:32:45Z,2025-11-03 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-11-03,7,"CMS, DHA, FDA, HHS, HRSA, IHS, OS",12.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2610,471298,2025-11-03T10:08:20Z,RITM0606742,64.62.156.67,1854216,Address,2025-07-28 17:15:21+00:00,2025-10-30T07:43:54Z,2025-11-03 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-11-03,7,"CDC, CMS, DHA, FDA, HHS, HRSA, OS",11.0
2629,471298,2025-11-03T10:08:20Z,RITM0606742,64.62.156.17,1871832,Address,2025-07-28 17:15:14+00:00,2025-10-30T07:43:54Z,2025-11-03 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-11-03,6,"CMS, DHA, FDA, HHS, HRSA, OS",11.0
2653,471298,2025-11-03T10:08:20Z,RITM0606742,64.62.156.20,1360645,Address,2025-08-13 15:08:38+00:00,2025-10-30T02:15:15Z,2025-11-03 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-11-03,6,"CMS, DHA, FDA, HHS, HRSA, OS",12.0
2771,471298,2025-11-03T10:08:20Z,TASK0891295,64.62.156.16,2456756,Address,2025-06-25 17:45:38+00:00,2025-10-29T23:00:10Z,2025-11-03 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-11-03,7,"CMS, DHA, FDA, HHS, HRSA, IHS, OS",11.0


In [ ]:
def excludeSoarData(recent_tags):
    # Extract unique indicators from recent_tags
    indicators = recent_tags['summary'].unique()

    attributes_data = []
    indicators_with_no_attributes = []

    for indicator in indicators:
        try:
            ro.set_http_method('GET')
            ro.set_request_uri(f'/v3/indicators/{indicator}?fields=attributes&resultStart=0&resultLimit=1000')
            response = tc.api_request(ro)

            if response.headers.get('content-type') == 'application/json':
                data = response.json().get('data', {})
                attributes = data.get('attributes', {}).get('data', [])

                if not attributes:
                    indicators_with_no_attributes.append(indicator)
                else:
                    for attr in attributes:
                        attr.update({
                            'id': data.get('id'),
                            'summary': data.get('summary'),
                            'type': data.get('type'),
                            'ownerName': data.get('ownerName')
                        })
                        attributes_data.append(attr)
        except Exception as e:
            if hasattr(e, 'response') and getattr(e.response, 'status_code', None) == 400:
                continue
            if "Status Code: 400" in str(e):
                continue
            pass

    attributes_observed_src = pd.json_normalize(attributes_data)

    if not attributes_observed_src.empty and attributes_observed_src['createdBy.lastName'].notnull().any():
        attributes_observed_src = attributes_observed_src[attributes_observed_src['createdBy.lastName'] != 'SOAR']

    attributes_observed_src = attributes_observed_src.drop_duplicates(subset='id').reset_index(drop=True)

    filtered_with_attrs = recent_tags[recent_tags['summary'].isin(attributes_observed_src['summary'])]
    no_attrs_df = recent_tags[recent_tags['summary'].isin(indicators_with_no_attributes)]

    filtered_recent_tags = pd.concat([filtered_with_attrs, no_attrs_df], ignore_index=True)
    filtered_recent_tags = filtered_recent_tags.drop_duplicates(subset='summary').reset_index(drop=True)
    
    return filtered_recent_tags

filtered_recent_tags = excludeSoarData(recent_tags)
display(filtered_recent_tags)

None

In [35]:
# Load threat assessment scores from Excel file
import pandas as pd

excel_file_path = r"Z:\HTOC\Data_Analytics\Data\Threat Assessment Scores\Threat_Assessment_Scores_20251103_084812.xlsx"

# Read the Excel file
threat_scores_df = pd.read_excel(excel_file_path, engine='openpyxl')

print(f"Loaded {len(threat_scores_df)} rows from the Excel file")
print(f"\nColumns in the DataFrame:")
print(threat_scores_df.columns.tolist())
print(f"\nFirst few rows:")
display(threat_scores_df)


Loaded 853 rows from the Excel file

Columns in the DataFrame:
['indicator', 'type', 'enrich_vtMaliciousCount', 'obs_count', 'rating', 'obs_penalty_multiplier', 'botnet_flag', 'falsePositives', 'Threat_Score', 'Severity', 'Explanation']

First few rows:


,indicator,type,enrich_vtMaliciousCount,obs_count,rating,obs_penalty_multiplier,botnet_flag,falsePositives,Threat_Score,Severity,Explanation
0,101.71.130.99,Address,10,19,3,0.998959,0,0,571,high,Severity: high. Top drivers: VT malicious (log...
1,101.89.174.236,Address,9,167,3,0.990849,0,0,516,high,Severity: high. Top drivers: VT malicious (log...
2,103.101.216.50,Address,0,3,4,0.999836,1,0,49,low,Severity: low. Top drivers: TC confidence (+1....
3,103.133.60.12,Address,1,43,1,0.997644,1,0,92,low,Severity: low. Top drivers: VT malicious (log-...
4,103.133.61.182,Address,2,29,1,0.998411,1,0,120,low,Severity: low. Top drivers: VT malicious (log-...
...,...,...,...,...,...,...,...,...,...,...,...
848,warlockstallioniso.com,Host,0,58,3,0.996822,1,0,53,low,Severity: low. Top drivers: TC confidence (+1....
849,www.deepseek.com,Host,0,260,3,0.985753,0,0,151,low,Severity: low. Top drivers: TC confidence (+1....
850,www.deepseek.com.cdn.cloudflare.net,Host,0,209,3,0.988548,0,0,148,low,Severity: low. Top drivers: TC confidence (+1....
851,www.sthda.com,Host,0,247,4,0.986466,0,0,97,low,Severity: low. Top drivers: TC confidence (+0....


In [ ]:
# Merge Threat_Score from Excel file into recent_tags
# Match on summary (recent_tags) and indicator (threat_scores_df)

# Select only the indicator and Threat_Score columns from the Excel data
threat_scores_subset = threat_scores_df[['indicator', 'Threat_Score']].copy()

# Merge Threat_Score into recent_tags by matching summary to indicator
filtered_recent_tags = filtered_recent_tags.merge(
    threat_scores_subset,
    left_on='summary',
    right_on='indicator',
    how='left'
)

# Drop the duplicate indicator column from the merge (keep summary)
if 'indicator' in filtered_recent_tags.columns:
    filtered_recent_tags.drop(columns=['indicator'], inplace=True, errors='ignore')

print(f"Merged Threat_Score into filtered_recent_tags")
display(filtered_recent_tags)


Merged Threat_Score into recent_tags


,id,lastUsed,description,summary,observations,type,dateAdded,lastModified,lastObserved,webLink,all_tags,observed_date,partner_count,partners,vtMaliciousCount,Threat_Score_x,Threat_Score_y,Threat_Score
0,30479,2025-11-02T14:39:01Z,185.220.101[.]182 is registered in Germany und...,185.220.101.182,228072,Address,2021-12-15 13:22:24+00:00,2025-11-03T12:55:25Z,2025-11-03 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[Zero Day, HHS Splunk Production API, HRSA Spl...",2025-11-03,1,CMS,13.0,504,504,504
1,471298,2025-11-03T10:08:20Z,RITM0582833,193.163.125.121,3609219,Address,2025-05-21 13:34:32+00:00,2025-11-03T12:55:04Z,2025-11-03 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, VA OIS CSOC CT...",2025-11-03,7,"CMS, DHA, FDA, HHS, HRSA, IHS, OS",13.0,573,573,573
2,471298,2025-11-03T10:08:20Z,Executive Summary: \n\n118.193.59[.]10 resolv...,118.193.59.10,11318565,Address,2025-02-12 19:49:28+00:00,2025-11-03T12:36:17Z,2025-11-03 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, Active Scanning: Scanning IP ...",2025-11-03,6,"CMS, DHA, FDA, HRSA, IHS, OS",12.0,384,384,384
3,35760,2025-11-03T14:02:40Z,INC9235915,165.154.173.226,577067,Address,2025-09-12 14:57:07+00:00,2025-11-03T12:36:17Z,2025-11-03 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-11-03,4,"CMS, FDA, IHS, OS",11.0,576,576,576
4,471298,2025-11-03T10:08:20Z,TASK0902923,198.235.24.194,15656064,Address,2025-07-28 17:34:11+00:00,2025-11-03T12:32:45Z,2025-11-03 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-11-03,7,"CMS, DHA, FDA, HHS, HRSA, IHS, OS",12.0,404,404,404
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75,471298,2025-11-03T10:08:20Z,RITM0606742,64.62.156.67,1854062,Address,2025-07-28 17:15:21+00:00,2025-10-30T07:43:54Z,2025-11-03 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-11-03,7,"CDC, CMS, DHA, FDA, HHS, HRSA, OS",11.0,387,387,387
76,471298,2025-11-03T10:08:20Z,RITM0606742,64.62.156.17,1871623,Address,2025-07-28 17:15:14+00:00,2025-10-30T07:43:54Z,2025-11-03 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-11-03,6,"CMS, DHA, FDA, HHS, HRSA, OS",11.0,398,398,398
77,471298,2025-11-03T10:08:20Z,RITM0606742,64.62.156.20,1360557,Address,2025-08-13 15:08:38+00:00,2025-10-30T02:15:15Z,2025-11-03 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-11-03,6,"CMS, DHA, FDA, HHS, HRSA, OS",12.0,408,408,408
78,471298,2025-11-03T10:08:20Z,TASK0891295,64.62.156.16,2456592,Address,2025-06-25 17:45:38+00:00,2025-10-29T23:00:10Z,2025-11-03 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-11-03,7,"CMS, DHA, FDA, HHS, HRSA, IHS, OS",11.0,388,388,388


In [ ]:
from IPython.display import display

# Get all unique partners (split by comma and flatten)
all_partners = set(
    p.strip()
    for partners in filtered_recent_tags['partners'].dropna().unique()
    for p in partners.split(',')
)

# Build buckets: each partner gets all rows where it appears in the partners column
partner_buckets = {
    partner: filtered_recent_tags[filtered_recent_tags['partners'].str.contains(rf'\b{partner}\b', na=False)]
    for partner in all_partners
}

# Show each partner's dataframe as a table
for partner, df in partner_buckets.items():
    print(f"Partner: {partner} ({len(df)} records)")
    display(df)

Partner: HHS (49 records)


,id,lastUsed,description,summary,observations,type,dateAdded,lastModified,lastObserved,webLink,all_tags,observed_date,partner_count,partners,vtMaliciousCount,Threat_Score_x,Threat_Score_y,Threat_Score
1,471298,2025-11-03T10:08:20Z,RITM0582833,193.163.125.121,3609219,Address,2025-05-21 13:34:32+00:00,2025-11-03T12:55:04Z,2025-11-03 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, VA OIS CSOC CT...",2025-11-03,7,"CMS, DHA, FDA, HHS, HRSA, IHS, OS",13.0,573,573,573
4,471298,2025-11-03T10:08:20Z,TASK0902923,198.235.24.194,15656064,Address,2025-07-28 17:34:11+00:00,2025-11-03T12:32:45Z,2025-11-03 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-11-03,7,"CMS, DHA, FDA, HHS, HRSA, IHS, OS",12.0,404,404,404
5,471298,2025-11-03T10:08:20Z,TASK0902923,205.210.31.66,16189313,Address,2025-07-28 17:15:57+00:00,2025-11-03T12:20:41Z,2025-11-03 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-11-03,7,"CMS, DHA, FDA, HHS, HRSA, IHS, OS",13.0,413,413,413
7,471298,2025-11-03T10:08:20Z,TASK0902923,198.235.24.68,15160328,Address,2025-07-28 17:15:37+00:00,2025-11-03T11:53:10Z,2025-11-03 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-11-03,7,"CMS, DHA, FDA, HHS, HRSA, IHS, OS",12.0,404,404,404
9,471298,2025-11-03T10:08:20Z,RITM0606742,65.49.1.92,1520224,Address,2025-07-28 17:34:12+00:00,2025-11-03T11:40:26Z,2025-11-03 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-11-03,6,"CMS, DHA, FDA, HHS, HRSA, OS",12.0,407,407,407
13,471298,2025-11-03T10:08:20Z,TASK0902923,65.49.1.50,1421826,Address,2025-07-28 17:16:02+00:00,2025-11-03T09:00:52Z,2025-11-03 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-11-03,7,"CMS, DHA, FDA, HHS, HRSA, IHS, OS",12.0,404,404,404
14,471298,2025-11-03T10:08:20Z,TASK0902923,205.210.31.212,15350949,Address,2025-07-28 17:34:02+00:00,2025-11-03T08:59:57Z,2025-11-03 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-11-03,7,"CMS, DHA, FDA, HHS, HRSA, IHS, OS",11.0,394,394,394
18,471298,2025-11-03T10:08:20Z,RITM0589984,170.64.134.89,1011652,Address,2025-06-16 17:42:50+00:00,2025-11-03T07:41:56Z,2025-11-03 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-11-03,5,"DHA, FDA, HHS, HRSA, OS",13.0,404,404,404
19,471298,2025-11-03T10:08:20Z,TASK0902923,205.210.31.230,15440209,Address,2025-07-28 17:34:08+00:00,2025-11-03T07:41:55Z,2025-11-03 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-11-03,7,"CMS, DHA, FDA, HHS, HRSA, IHS, OS",11.0,394,394,394
20,471298,2025-11-03T10:08:20Z,TASK0902923,198.235.24.89,15351092,Address,2025-07-28 17:16:00+00:00,2025-11-03T07:41:55Z,2025-11-03 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-11-03,7,"CMS, DHA, FDA, HHS, HRSA, IHS, OS",11.0,394,394,394


Partner: CMS (69 records)


,id,lastUsed,description,summary,observations,type,dateAdded,lastModified,lastObserved,webLink,all_tags,observed_date,partner_count,partners,vtMaliciousCount,Threat_Score_x,Threat_Score_y,Threat_Score
0,30479,2025-11-02T14:39:01Z,185.220.101[.]182 is registered in Germany und...,185.220.101.182,228072,Address,2021-12-15 13:22:24+00:00,2025-11-03T12:55:25Z,2025-11-03 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[Zero Day, HHS Splunk Production API, HRSA Spl...",2025-11-03,1,CMS,13.0,504,504,504
1,471298,2025-11-03T10:08:20Z,RITM0582833,193.163.125.121,3609219,Address,2025-05-21 13:34:32+00:00,2025-11-03T12:55:04Z,2025-11-03 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, VA OIS CSOC CT...",2025-11-03,7,"CMS, DHA, FDA, HHS, HRSA, IHS, OS",13.0,573,573,573
2,471298,2025-11-03T10:08:20Z,Executive Summary: \n\n118.193.59[.]10 resolv...,118.193.59.10,11318565,Address,2025-02-12 19:49:28+00:00,2025-11-03T12:36:17Z,2025-11-03 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, Active Scanning: Scanning IP ...",2025-11-03,6,"CMS, DHA, FDA, HRSA, IHS, OS",12.0,384,384,384
3,35760,2025-11-03T14:02:40Z,INC9235915,165.154.173.226,577067,Address,2025-09-12 14:57:07+00:00,2025-11-03T12:36:17Z,2025-11-03 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-11-03,4,"CMS, FDA, IHS, OS",11.0,576,576,576
4,471298,2025-11-03T10:08:20Z,TASK0902923,198.235.24.194,15656064,Address,2025-07-28 17:34:11+00:00,2025-11-03T12:32:45Z,2025-11-03 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-11-03,7,"CMS, DHA, FDA, HHS, HRSA, IHS, OS",12.0,404,404,404
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
73,471298,2025-11-03T10:08:20Z,INC9263763,193.32.162.145,842357,Address,2025-10-02 17:13:20+00:00,2025-10-31T02:34:38Z,2025-11-03 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-11-03,4,"CMS, DHA, FDA, OS",14.0,621,621,621
75,471298,2025-11-03T10:08:20Z,RITM0606742,64.62.156.67,1854062,Address,2025-07-28 17:15:21+00:00,2025-10-30T07:43:54Z,2025-11-03 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-11-03,7,"CDC, CMS, DHA, FDA, HHS, HRSA, OS",11.0,387,387,387
76,471298,2025-11-03T10:08:20Z,RITM0606742,64.62.156.17,1871623,Address,2025-07-28 17:15:14+00:00,2025-10-30T07:43:54Z,2025-11-03 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-11-03,6,"CMS, DHA, FDA, HHS, HRSA, OS",11.0,398,398,398
77,471298,2025-11-03T10:08:20Z,RITM0606742,64.62.156.20,1360557,Address,2025-08-13 15:08:38+00:00,2025-10-30T02:15:15Z,2025-11-03 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-11-03,6,"CMS, DHA, FDA, HHS, HRSA, OS",12.0,408,408,408


Partner: DHA (45 records)


...

,id,lastUsed,description,summary,observations,type,dateAdded,lastModified,lastObserved,webLink,all_tags,observed_date,partner_count,partners,vtMaliciousCount,Threat_Score_x,Threat_Score_y,Threat_Score
21,471298,2025-11-03T10:08:20Z,TASK0902923,198.235.24.200,15766425,Address,2025-07-28 17:15:50+00:00,2025-11-03T07:41:55Z,2025-11-03 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-11-03,8,"CDC, CMS, DHA, FDA, HHS, HRSA, IHS, OS",11.0,394,394,394
75,471298,2025-11-03T10:08:20Z,RITM0606742,64.62.156.67,1854062,Address,2025-07-28 17:15:21+00:00,2025-10-30T07:43:54Z,2025-11-03 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-11-03,7,"CDC, CMS, DHA, FDA, HHS, HRSA, OS",11.0,387,387,387


Partner: HRSA (51 records)


,id,lastUsed,description,summary,observations,type,dateAdded,lastModified,lastObserved,webLink,all_tags,observed_date,partner_count,partners,vtMaliciousCount,Threat_Score_x,Threat_Score_y,Threat_Score
1,471298,2025-11-03T10:08:20Z,RITM0582833,193.163.125.121,3609219,Address,2025-05-21 13:34:32+00:00,2025-11-03T12:55:04Z,2025-11-03 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, VA OIS CSOC CT...",2025-11-03,7,"CMS, DHA, FDA, HHS, HRSA, IHS, OS",13.0,573,573,573
2,471298,2025-11-03T10:08:20Z,Executive Summary: \n\n118.193.59[.]10 resolv...,118.193.59.10,11318565,Address,2025-02-12 19:49:28+00:00,2025-11-03T12:36:17Z,2025-11-03 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, Active Scanning: Scanning IP ...",2025-11-03,6,"CMS, DHA, FDA, HRSA, IHS, OS",12.0,384,384,384
4,471298,2025-11-03T10:08:20Z,TASK0902923,198.235.24.194,15656064,Address,2025-07-28 17:34:11+00:00,2025-11-03T12:32:45Z,2025-11-03 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-11-03,7,"CMS, DHA, FDA, HHS, HRSA, IHS, OS",12.0,404,404,404
5,471298,2025-11-03T10:08:20Z,TASK0902923,205.210.31.66,16189313,Address,2025-07-28 17:15:57+00:00,2025-11-03T12:20:41Z,2025-11-03 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-11-03,7,"CMS, DHA, FDA, HHS, HRSA, IHS, OS",13.0,413,413,413
7,471298,2025-11-03T10:08:20Z,TASK0902923,198.235.24.68,15160328,Address,2025-07-28 17:15:37+00:00,2025-11-03T11:53:10Z,2025-11-03 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-11-03,7,"CMS, DHA, FDA, HHS, HRSA, IHS, OS",12.0,404,404,404
9,471298,2025-11-03T10:08:20Z,RITM0606742,65.49.1.92,1520224,Address,2025-07-28 17:34:12+00:00,2025-11-03T11:40:26Z,2025-11-03 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-11-03,6,"CMS, DHA, FDA, HHS, HRSA, OS",12.0,407,407,407
13,471298,2025-11-03T10:08:20Z,TASK0902923,65.49.1.50,1421826,Address,2025-07-28 17:16:02+00:00,2025-11-03T09:00:52Z,2025-11-03 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-11-03,7,"CMS, DHA, FDA, HHS, HRSA, IHS, OS",12.0,404,404,404
14,471298,2025-11-03T10:08:20Z,TASK0902923,205.210.31.212,15350949,Address,2025-07-28 17:34:02+00:00,2025-11-03T08:59:57Z,2025-11-03 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-11-03,7,"CMS, DHA, FDA, HHS, HRSA, IHS, OS",11.0,394,394,394
18,471298,2025-11-03T10:08:20Z,RITM0589984,170.64.134.89,1011652,Address,2025-06-16 17:42:50+00:00,2025-11-03T07:41:56Z,2025-11-03 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-11-03,5,"DHA, FDA, HHS, HRSA, OS",13.0,404,404,404
19,471298,2025-11-03T10:08:20Z,TASK0902923,205.210.31.230,15440209,Address,2025-07-28 17:34:08+00:00,2025-11-03T07:41:55Z,2025-11-03 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-11-03,7,"CMS, DHA, FDA, HHS, HRSA, IHS, OS",11.0,394,394,394


Partner: FDA (57 records)


,id,lastUsed,description,summary,observations,type,dateAdded,lastModified,lastObserved,webLink,all_tags,observed_date,partner_count,partners,vtMaliciousCount,Threat_Score_x,Threat_Score_y,Threat_Score
1,471298,2025-11-03T10:08:20Z,RITM0582833,193.163.125.121,3609219,Address,2025-05-21 13:34:32+00:00,2025-11-03T12:55:04Z,2025-11-03 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, VA OIS CSOC CT...",2025-11-03,7,"CMS, DHA, FDA, HHS, HRSA, IHS, OS",13.0,573,573,573
2,471298,2025-11-03T10:08:20Z,Executive Summary: \n\n118.193.59[.]10 resolv...,118.193.59.10,11318565,Address,2025-02-12 19:49:28+00:00,2025-11-03T12:36:17Z,2025-11-03 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, Active Scanning: Scanning IP ...",2025-11-03,6,"CMS, DHA, FDA, HRSA, IHS, OS",12.0,384,384,384
3,35760,2025-11-03T14:02:40Z,INC9235915,165.154.173.226,577067,Address,2025-09-12 14:57:07+00:00,2025-11-03T12:36:17Z,2025-11-03 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-11-03,4,"CMS, FDA, IHS, OS",11.0,576,576,576
4,471298,2025-11-03T10:08:20Z,TASK0902923,198.235.24.194,15656064,Address,2025-07-28 17:34:11+00:00,2025-11-03T12:32:45Z,2025-11-03 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-11-03,7,"CMS, DHA, FDA, HHS, HRSA, IHS, OS",12.0,404,404,404
5,471298,2025-11-03T10:08:20Z,TASK0902923,205.210.31.66,16189313,Address,2025-07-28 17:15:57+00:00,2025-11-03T12:20:41Z,2025-11-03 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-11-03,7,"CMS, DHA, FDA, HHS, HRSA, IHS, OS",13.0,413,413,413
7,471298,2025-11-03T10:08:20Z,TASK0902923,198.235.24.68,15160328,Address,2025-07-28 17:15:37+00:00,2025-11-03T11:53:10Z,2025-11-03 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-11-03,7,"CMS, DHA, FDA, HHS, HRSA, IHS, OS",12.0,404,404,404
9,471298,2025-11-03T10:08:20Z,RITM0606742,65.49.1.92,1520224,Address,2025-07-28 17:34:12+00:00,2025-11-03T11:40:26Z,2025-11-03 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-11-03,6,"CMS, DHA, FDA, HHS, HRSA, OS",12.0,407,407,407
13,471298,2025-11-03T10:08:20Z,TASK0902923,65.49.1.50,1421826,Address,2025-07-28 17:16:02+00:00,2025-11-03T09:00:52Z,2025-11-03 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-11-03,7,"CMS, DHA, FDA, HHS, HRSA, IHS, OS",12.0,404,404,404
14,471298,2025-11-03T10:08:20Z,TASK0902923,205.210.31.212,15350949,Address,2025-07-28 17:34:02+00:00,2025-11-03T08:59:57Z,2025-11-03 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-11-03,7,"CMS, DHA, FDA, HHS, HRSA, IHS, OS",11.0,394,394,394
16,35057,2025-10-31T22:44:21Z,TASK0912447,192.42.116.178,142695,Address,2025-06-16 17:42:12+00:00,2025-11-03T08:33:39Z,2025-11-03 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[OS Splunk API, FDA Splunk API, CMS Splunk API...",2025-11-03,1,FDA,14.0,607,607,607


In [22]:
import os
import re
import time
from datetime import datetime
import pandas as pd
import win32com.client as win32

Tippers_Path = r'C:\Users\jaskew\Documents\Test Outputs\Tippers'
os.makedirs(Tippers_Path, exist_ok=True)

# Add today's date to the filename
today_str = datetime.utcnow().strftime("%Y%m%d")
excel_path = os.path.join(Tippers_Path, f"tippers_by_partner_{today_str}.xlsx")

with pd.ExcelWriter(excel_path, engine='xlsxwriter') as writer:
    for partner, df in partner_buckets.items():
        # Excel sheet names can't be longer than 31 chars or contain special chars
        safe_partner = re.sub(r'[^a-zA-Z0-9_-]', '_', partner)[:31]

        # Convert all timezone-aware datetime columns to naive
        for col in df.select_dtypes(include=['datetimetz']).columns:
            df[col] = df[col].dt.tz_localize(None)

        # Write to Excel first
        df.to_excel(writer, sheet_name=safe_partner, index=False)

        # Then get the worksheet object to format
        worksheet = writer.sheets[safe_partner]
        worksheet.autofilter(0, 0, len(df), len(df.columns) - 1)
        worksheet.freeze_panes(1, 0)

        for i, col in enumerate(df.columns):
            # Set width based on max text length in the column, or column name
            max_len = max(
                df[col].astype(str).map(len).max() if not df.empty else 0,
                len(str(col))
            )
            worksheet.set_column(i, i, min(max_len + 2, 50))  # Cap width at 50

print(f"Excel file with partner tabs saved to: {excel_path}")


C:\Users\jaskew\AppData\Local\Temp\ipykernel_640\786895275.py:12: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  today_str = datetime.utcnow().strftime("%Y%m%d")
C:\Users\jaskew\AppData\Local\Temp\ipykernel_640\786895275.py:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[col] = df[col].dt.tz_localize(None)
C:\Users\jaskew\AppData\Local\Temp\ipykernel_640\786895275.py:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas

Excel file with partner tabs saved to: C:\Users\jaskew\Documents\Test Outputs\Tippers\tippers_by_partner_20251103.xlsx
